# Módulo 08 — Base `ecomplexity` y gráficas del ICE municipal

Este notebook hace dos cosas:

1. **Exporta la base municipio × rama** con las variables del paquete `ecomplexity`
   (`año, clave_municipio, nom_municipio, rama, diversity, ubiquity, mcp, eci, pci, density, coi, cog, rca`)
   para el **ICE final** (UE, rama, núcleo 30/5/10, PCI espectral y proyección).
   La lógica está en `codigo/ice/export_ecomplexity.py`. En el núcleo coincide con `ecomplexity` (corr = 1).
2. **Genera todas las gráficas** organizadas por tipo y en ciclo sobre los años donde aplica.
   Se guardan en `figuras/ice/` (y las del apéndice en `figuras/ice/anexo/`).

**Estructura del código (modular):**
- Celda 0: configuración, idioma (`IDIOMA = "en"` para el artículo en inglés, `"es"` para español) y las
  funciones comunes de diseño: `T()` (texto bilingüe), `finalizar()` (título centrado y nota al pie ajustada
  al ancho de la gráfica) y `guardar()`.
- Cada sección: (a) una función `graficar_*` con el diseño y (b) una celda de ejecución con el ciclo de años.
  Para cambiar textos, colores o tamaños, edita la función; para cambiar años o municipios, la celda de ejecución.

**Reglas de diseño comunes (implementadas en `finalizar()`):**
- El título se centra sobre el área principal de la gráfica (o sobre todo el contenido) y mide 10% más que la
  base de 13 pt (`TAM_TITULO = 14.3`); los mapas y la matriz regional usan un título del doble de tamaño.
- La fuente y las notas se alinean a la izquierda y se ajustan con `textwrap` al ancho de la gráfica + leyenda:
  el ancho de envoltura se reduce hasta que el texto renderizado cabe, por lo que nunca se corta a los lados.
- La imagen se recorta de forma simétrica alrededor del área de referencia, de modo que la gráfica queda centrada.

**Requisitos:** haber corrido los Módulos 01–07 (`python -m codigo.ice.run_all`). Paquetes: numpy, pandas,
scipy, matplotlib, seaborn, networkx, pyshp (no requiere geopandas: los mapas se dibujan con `pyshp` + `matplotlib`).
Los nombres de rama en inglés están en `codigo/ice/catalogo_ramas_en.csv` (traducción propia de los títulos SCIAN,
siguiendo la terminología de NAICS).

## 0. Configuración y funciones comunes de diseño

In [1]:
import math
import sys
import textwrap
import warnings
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import shapefile
from matplotlib.collections import PolyCollection
from matplotlib.transforms import Bbox
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore", category=FutureWarning)

# Raíz del repositorio: la carpeta que contiene data/ y codigo/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "codigo" / "ice").exists())
sys.path.insert(0, str(ROOT))
from codigo.ice import config as C  # noqa: E402

FIG = ROOT / "figuras" / "ice"
TAB = FIG / "tablas"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

ANIOS = C.ANIOS            # [2003, 2008, 2013, 2018, 2023]
DPI = 300
REGENERAR_BASE = False     # True para volver a calcular la base ecomplexity aunque exista
IDIOMA = "en"              # "en" (artículo en inglés) o "es"

TAM_BASE_TITULO = 13
TAM_TITULO = TAM_BASE_TITULO * 1.1      # +10% sobre la base
TAM_PIE = 9.5
GAP_PIE = 0.30                           # pulgadas entre el contenido y la nota al pie
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.titlesize": 12, "axes.labelsize": 11})


def T(es, en):
    """Devuelve el texto en el idioma activo."""
    return en if IDIOMA == "en" else es


FUENTE = T("Fuente: elaboración propia con datos de los Censos Económicos 2004-2024 (INEGI), años de referencia 2003-2023.",
           "Source: authors' elaboration with data from INEGI's Economic Censuses 2004-2024 (reference years 2003-2023).")
NOTA_ICE = T("El ICE se calcula con unidades económicas (UE) por rama SCIAN (4 dígitos); el PCI se estima en un núcleo de "
             "municipios y ramas (UE totales >= 30, >= 5 ramas; ramas presentes en >= 10 municipios) y el ICE se proyecta a "
             "todos los municipios. Estandarizado con la media y la desviación estándar del núcleo de cada año.",
             "The ECI is computed from the number of economic units (establishments) by SCIAN industry group (4 digits); "
             "the PCI is estimated on a core of municipalities and industries (>= 30 establishments, >= 5 industries; "
             "industries present in >= 10 municipalities) and the ECI is projected to all municipalities. Standardized with "
             "the core mean and standard deviation of each year.")
ICE = T("ICE", "ECI")
NOM_ICE = T("Índice de Complejidad Económica (ICE)", "Economic Complexity Index (ECI)")
NOM_PCI = T("PCI (complejidad de la rama)", "PCI (industry complexity)")
NOM_DIST = T("Distancia tecnológica (1 - densidad)", "Technological distance (1 - density)")
MEXICO = T("México", "Mexico")

REGION_CEEY = {
    "02": "Norte", "05": "Norte", "08": "Norte", "19": "Norte", "26": "Norte", "28": "Norte",
    "03": "Norte-occidente", "10": "Norte-occidente", "18": "Norte-occidente", "25": "Norte-occidente",
    "32": "Norte-occidente",
    "01": "Centro-norte", "06": "Centro-norte", "14": "Centro-norte", "16": "Centro-norte", "24": "Centro-norte",
    "09": "Centro", "11": "Centro", "13": "Centro", "15": "Centro", "17": "Centro", "21": "Centro",
    "22": "Centro", "29": "Centro",
    "04": "Sur", "07": "Sur", "12": "Sur", "20": "Sur", "23": "Sur", "27": "Sur", "30": "Sur", "31": "Sur",
}
ORDEN_REGIONES = ["Norte", "Norte-occidente", "Centro-norte", "Centro", "Sur"]
NOMBRE_REGION = {"Norte": T("Norte", "North"), "Norte-occidente": T("Norte-occidente", "North-West"),
                 "Centro-norte": T("Centro-norte", "Center-North"), "Centro": T("Centro", "Center"),
                 "Sur": T("Sur", "South"), "Nacional": T("Nacional", "National")}


# ------------------------------------------------------------------ diseño común
def _envolver(txt, texto, ancho_pulg, renderer):
    """Envuelve `texto` (párrafos separados por \n) hasta que su ancho renderizado sea <= ancho_pulg."""
    parrafos = texto.split("\n")
    n = max(len(p) for p in parrafos)
    while True:
        txt.set_text("\n".join(textwrap.fill(p, n, break_long_words=False) for p in parrafos))
        if txt.get_window_extent(renderer).width / txt.figure.dpi <= 0.98 * ancho_pulg or n <= 15:
            return
        n -= 1


def finalizar(fig, titulo, pie, ref=None, tam_titulo=None, tam_pie=None, gap_pie=GAP_PIE, gap_titulo=0.18):
    """Coloca el título y la nota al pie y fija el recorte simétrico de la imagen.

    ref: ejes (o lista de ejes) sobre los que se centra la figura. None = centra todo el contenido.
    El título se centra en ese eje de simetría; la nota se alinea a la izquierda del contenido y se
    envuelve al ancho del contenido (gráfica + leyenda + barra de color).
    """
    fig.canvas.draw()
    fig.set_layout_engine("none")          # congela la disposición antes de añadir textos externos
    r = fig.canvas.get_renderer()
    inv = fig.dpi_scale_trans.inverted()
    bb = fig.get_tightbbox(r)              # contenido, en pulgadas
    if ref is None:
        cx = (bb.x0 + bb.x1) / 2
    else:
        refs = ref if isinstance(ref, (list, tuple, np.ndarray)) else [ref]
        rb = Bbox.union([a.get_window_extent(r) for a in np.ravel(refs)]).transformed(inv)
        cx = (rb.x0 + rb.x1) / 2
    medio = max(cx - bb.x0, bb.x1 - cx)
    W, H = fig.get_figwidth(), fig.get_figheight()
    t = fig.text(cx / W, (bb.y1 + gap_titulo) / H, "", ha="center", va="bottom", multialignment="center",
                 fontsize=tam_titulo or TAM_TITULO, fontweight="bold")
    _envolver(t, titulo, 2 * medio, r)
    p = fig.text(bb.x0 / W, (bb.y0 - gap_pie) / H, "", ha="left", va="top", fontsize=tam_pie or TAM_PIE,
                 linespacing=1.45)
    _envolver(p, pie, bb.width, r)
    tb, pb = t.get_window_extent(r).transformed(inv), p.get_window_extent(r).transformed(inv)
    m = 0.12
    fig._bbox_guardar = Bbox([[cx - medio - m, pb.y0 - m], [cx + medio + m, tb.y1 + m]])
    return fig


def guardar(fig, nombre):
    """Guarda en figuras/ice/ a 300 dpi con el recorte fijado por finalizar() y muestra la figura."""
    ruta = FIG / f"{nombre}.png"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(ruta, dpi=DPI, bbox_inches=getattr(fig, "_bbox_guardar", "tight"), facecolor="white")
    print(f"Guardado: {ruta.relative_to(ROOT)}")
    plt.show()
    plt.close(fig)


def pie_texto(*notas):
    """Fuente + notas en párrafos separados."""
    return FUENTE + "".join(f"\n{T('Notas', 'Notes')}: {n}" if i == 0 else f" {n}" for i, n in enumerate(notas))

## 1. Exportación de la base `ecomplexity` (insumo de todas las gráficas)

In [2]:
BASES_ECO = [C.CLEAN / f"ecomplexity_rama_ue_{a}.csv.gz" for a in ANIOS]   # un archivo por año
if REGENERAR_BASE or not all(f.exists() for f in BASES_ECO):
    from codigo.ice import export_ecomplexity
    export_ecomplexity.main()

eco = pd.concat([pd.read_csv(f, dtype={"clave_municipio": str, "rama": str}) for f in BASES_ECO],
                ignore_index=True)
ramas = pd.read_csv(C.CLEAN / "catalogo_ramas.csv", dtype={"rama": str})
ramas_en = pd.read_csv(ROOT / "codigo" / "ice" / "catalogo_ramas_en.csv", dtype={"rama": str})[["rama", "nombre_rama_en"]]
ramas = ramas.merge(ramas_en, on="rama", how="left")
ramas["nombre"] = ramas.nombre_rama_en if IDIOMA == "en" else ramas.nombre_rama
prox = pd.read_csv(C.CLEAN / "proximidad_ramas.csv.gz", dtype={"rama_1": str, "rama_2": str})
eco["distancia"] = 1 - eco["density"]
eco = eco.merge(ramas[["rama", "nombre_rama", "nombre"]], on="rama", how="left")

# Una fila por municipio-año
mun = (eco.drop_duplicates(["año", "clave_municipio"])
          [["año", "clave_municipio", "nom_municipio", "eci", "coi", "diversity", "en_nucleo"]]
          .reset_index(drop=True))
mun["cve_ent"] = mun.clave_municipio.str[:2]
mun["region_ceey"] = mun.cve_ent.map(REGION_CEEY)
# Una fila por rama-año
prod = (eco.drop_duplicates(["año", "rama"])[["año", "rama", "nombre_rama", "nombre", "pci", "ubiquity"]]
           .reset_index(drop=True))

print(f"Base ecomplexity: {len(eco):,} filas | municipios por año: "
      f"{mun.groupby('año').size().to_dict()} | ramas por año: {prod.groupby('año').size().to_dict()}")
eco.head()

Base ecomplexity: 3,286,998 filas | municipios por año: {2003: 2447, 2008: 2455, 2013: 2456, 2018: 2465, 2023: 2469} | ramas por año: {2003: 261, 2008: 269, 2013: 267, 2018: 269, 2023: 271}


    año  ...                      nombre
0  2003  ...                 Aquaculture
1  2003  ...                     Fishing
2  2003  ...      Oil and gas extraction
3  2003  ...            Metal ore mining
4  2003  ...  Nonmetallic mineral mining

[5 rows x 17 columns]

## 2. Tablas: municipios y ramas más y menos complejos (por año)

In [3]:
def tabla_ranking_municipios(anio, n=10):
    d = mun[mun.año == anio].sort_values("eci", ascending=False)
    cols = ["clave_municipio", "nom_municipio", "eci", "diversity", "en_nucleo"]
    return d.head(n)[cols].assign(grupo="top"), d.tail(n)[cols].iloc[::-1].assign(grupo="bottom")


def tabla_ranking_ramas(anio, n=10):
    d = prod[prod.año == anio].sort_values("pci", ascending=False)
    cols = ["rama", "nombre_rama", "nombre", "pci", "ubiquity"]
    return d.head(n)[cols].assign(grupo="top"), d.tail(n)[cols].iloc[::-1].assign(grupo="bottom")


tm, tr = [], []
for anio in ANIOS:
    tm += [t.assign(año=anio) for t in tabla_ranking_municipios(anio)]
    tr += [t.assign(año=anio) for t in tabla_ranking_ramas(anio)]
tm, tr = pd.concat(tm), pd.concat(tr)
tm.to_csv(TAB / "ranking_municipios_ice.csv", index=False)
tr.to_csv(TAB / "ranking_ramas_pci.csv", index=False)
print("Guardado: figuras/ice/tablas/ranking_municipios_ice.csv, ranking_ramas_pci.csv")
tm[tm.año == 2023]

Guardado: figuras/ice/tablas/ranking_municipios_ice.csv, ranking_ramas_pci.csv


      clave_municipio             nom_municipio  ...   grupo   año
10786           19006                   Apodaca  ...     top  2023
10100           09010            Álvaro Obregón  ...     top  2023
11632           22014                 Querétaro  ...     top  2023
10106           09016            Miguel Hidalgo  ...     top  2023
11844           28032                   Reynosa  ...     top  2023
10061           08037                    Juárez  ...     top  2023
10104           09014             Benito Juárez  ...     top  2023
10819           19039                 Monterrey  ...     top  2023
10799           19019    San Pedro Garza García  ...     top  2023
10590           15104       Tlalnepantla de Baz  ...     top  2023
10977           20146    San Francisco Logueche  ...  bottom  2023
11142           20311    San Pedro Jaltepetongo  ...  bottom  2023
11355           20524    Santo Domingo Yodohino  ...  bottom  2023
11399           20568         Zapotitlán Palmas  ...  bottom  

## 3. Distribución del ICE por año (histograma + KDE)

Todas las gráficas anuales comparten el mismo límite del eje Y (`YLIM_G01`), calculado como el máximo de la
densidad del histograma y del KDE entre los cinco años.

In [4]:
BINS_G01 = 40


def ylim_distribucion(bins=BINS_G01):
    """Límite común del eje Y: máximo del histograma (densidad) y del KDE en todos los años."""
    m = 0
    for a in ANIOS:
        s = mun.loc[mun.año == a, "eci"].dropna().to_numpy()
        h, _ = np.histogram(s, bins=bins, density=True)
        k = gaussian_kde(s)(np.linspace(s.min(), s.max(), 512))
        m = max(m, h.max(), k.max())
    return (0, 1.08 * m)


def graficar_distribucion_ice(anio, ylim):
    s = mun.loc[mun.año == anio, "eci"].dropna()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    sns.histplot(s, kde=True, stat="density", bins=BINS_G01, color="navy", ax=ax)
    ax.axvline(0, color="gray", ls="--", lw=0.8)
    ax.set_ylim(ylim)
    ax.set_xlabel(NOM_ICE, fontweight="bold")
    ax.set_ylabel(T("Densidad", "Density"), fontweight="bold")
    ax.text(0.98, 0.95, f"n = {len(s):,}\n{T('asimetría', 'skewness')} = {s.skew():.2f}\n"
            f"{T('curtosis', 'kurtosis')} = {s.kurtosis():.2f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9, bbox=dict(facecolor="white", edgecolor="lightgray"))
    titulo = T(f"Distribución del Índice de Complejidad Económica a nivel municipal, México, {anio}",
               f"Distribution of the municipal Economic Complexity Index, Mexico, {anio}")
    return finalizar(fig, titulo, pie_texto(NOTA_ICE), ref=ax)


def graficar_distribucion_ice_todos():
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for anio, color in zip(ANIOS, sns.color_palette("viridis", len(ANIOS))):
        sns.kdeplot(mun.loc[mun.año == anio, "eci"], ax=ax, label=str(anio), color=color, lw=2)
    ax.set_xlabel(NOM_ICE, fontweight="bold")
    ax.set_ylabel(T("Densidad", "Density"), fontweight="bold")
    ax.legend(title=T("Año", "Year"))
    titulo = T("Distribución del ICE municipal por año censal, México, 2003-2023",
               "Distribution of the municipal ECI by census year, Mexico, 2003-2023")
    return finalizar(fig, titulo, pie_texto(NOTA_ICE, T("El ICE es relativo a cada año: las curvas comparan formas, no niveles.",
                                                        "The ECI is relative to each year: curves compare shapes, not levels.")),
                     ref=ax)

In [5]:
YLIM_G01 = ylim_distribucion()
for anio in ANIOS:
    guardar(graficar_distribucion_ice(anio, YLIM_G01), f"G01_distribucion_ice_{anio}")
guardar(graficar_distribucion_ice_todos(), "G01_distribucion_ice_todos_los_anios")

Guardado: figuras/ice/G01_distribucion_ice_2003.png
Guardado: figuras/ice/G01_distribucion_ice_2008.png
Guardado: figuras/ice/G01_distribucion_ice_2013.png
Guardado: figuras/ice/G01_distribucion_ice_2018.png
Guardado: figuras/ice/G01_distribucion_ice_2023.png
Guardado: figuras/ice/G01_distribucion_ice_todos_los_anios.png


## 4. Diagrama distancia–PCI de los municipios más y menos complejos (por año)

- **Eje Y homogéneo:** todos los años (y G02 y G03) usan el mismo rango de PCI (`YLIM_PCI`).
- **Barra de color lineal:** `Normalize` lineal (no divergente), de modo que las marcas de la barra quedan
  equiespaciadas. G02 usa 0 a `VMAX_RCA_TOP` (común a todos los años, percentil 95 del RCA de los cinco municipios
  más complejos en todos los años, redondeado hacia arriba); G03 usa 0.0 a 1.0. Los valores superiores al máximo se
  pintan con el color del extremo (flecha `extend="max"`). La línea roja en la barra marca el umbral RCA = 1.

In [6]:
def top_municipios(anio, n_mun, extremo):
    return mun[mun.año == anio].sort_values("eci", ascending=(extremo == "bottom")).head(n_mun)


def graficar_distancia_pci_municipios(anio, n_mun=5, extremo="top", vmax=1.0, ylim=None, cmap="viridis"):
    m = top_municipios(anio, n_mun, extremo)
    d = eco[(eco.año == anio) & eco.clave_municipio.isin(m.clave_municipio)]
    norm = mcolors.Normalize(vmin=0, vmax=vmax)
    fig, ax = plt.subplots(figsize=(10, 7))
    sc = ax.scatter(d.distancia, d.pci, c=d.rca.clip(upper=vmax), cmap=cmap, norm=norm,
                    s=d.rca.clip(0.1, 10) * 40 + 20, edgecolor="black", linewidth=0.5, alpha=0.85)
    paso = 1 if vmax > 2 else 0.2
    cb = fig.colorbar(sc, ax=ax, extend="max", ticks=np.round(np.arange(0, vmax + 1e-9, paso), 1))
    cb.set_label(T("RCA (ventaja comparativa revelada)", "RCA (revealed comparative advantage)"), fontsize=11)
    if vmax > 1:
        cb.ax.axhline(1, color="red", lw=1.5)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_xlabel(NOM_DIST, fontweight="bold")
    ax.set_ylabel(NOM_PCI, fontweight="bold")
    ax.grid(True, ls="--", alpha=0.5)
    texto = T("mayor" if extremo == "top" else "menor", "highest" if extremo == "top" else "lowest")
    titulo = T(f"Distancia tecnológica y complejidad de las ramas (PCI) de los {n_mun} municipios con {texto} ICE, México, {anio}",
               f"Technological distance and industry complexity (PCI) for the {n_mun} municipalities with the {texto} ECI, Mexico, {anio}")
    nombres = ", ".join(f"{r.nom_municipio} ({r.clave_municipio}, {ICE} {r.eci:.2f})" for r in m.itertuples())
    escala = (T(f"Escala de color lineal de 0 a {vmax:g}; los valores mayores se muestran con el color del extremo.",
                f"Linear color scale from 0 to {vmax:g}; larger values are shown with the end color.")
              + T(" La línea roja marca RCA = 1.", " The red line marks RCA = 1.") * (vmax > 1))
    pie = pie_texto(T("cada punto es una combinación municipio-rama; el tamaño y el color reflejan el RCA.",
                      "each point is a municipality-industry pair; size and color reflect the RCA."), escala,
                    T(f"Municipios: {nombres}.", f"Municipalities: {nombres}."))
    return finalizar(fig, titulo, pie)

In [7]:
YLIM_PCI = (math.floor(prod.pci.min() - 0.5), math.ceil(prod.pci.max() + 0.5))
VMAX_RCA_TOP = math.ceil(max(eco[(eco.año == a) & eco.clave_municipio.isin(top_municipios(a, 5, "top").clave_municipio)]
                             .rca.quantile(0.95) for a in ANIOS))
VMAX_RCA_BOTTOM = 1.0
print(f"Eje Y común (PCI): {YLIM_PCI} | barra de color G02: 0-{VMAX_RCA_TOP} | G03: 0-{VMAX_RCA_BOTTOM}")
for anio in ANIOS:
    guardar(graficar_distancia_pci_municipios(anio, extremo="top", vmax=VMAX_RCA_TOP, ylim=YLIM_PCI),
            f"G02_distancia_pci_top5_municipios_{anio}")
    guardar(graficar_distancia_pci_municipios(anio, extremo="bottom", vmax=VMAX_RCA_BOTTOM, ylim=YLIM_PCI),
            f"G03_distancia_pci_bottom5_municipios_{anio}")

Eje Y común (PCI): (-5, 10) | barra de color G02: 0-9 | G03: 0-1.0
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2003.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2003.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2008.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2008.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2013.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2013.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2018.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2018.png
Guardado: figuras/ice/G02_distancia_pci_top5_municipios_2023.png
Guardado: figuras/ice/G03_distancia_pci_bottom5_municipios_2023.png


## 5. Ramas más y menos complejas: PCI y distancia de los municipios (por año)

Las etiquetas de rama se envuelven con el mismo ancho máximo (`ANCHO_ETIQUETA_RAMA` caracteres) y continúan en el
renglón siguiente, sin truncarse. Las etiquetas de ubicuidad se mantienen dentro de la caja: si alguna se sale,
se amplía el eje X. La nota al pie se separa 50% más que en el resto de las gráficas (`GAP_PIE * 1.5`).

In [8]:
ANCHO_ETIQUETA_RAMA = 30


def etiqueta_rama(r, ancho=ANCHO_ETIQUETA_RAMA):
    nom = r.nombre if isinstance(r.nombre, str) else ""
    return textwrap.fill(f"{r.rama} {nom}", ancho, break_long_words=False)


def textos_dentro(ax, textos, max_iter=20):
    """Amplía el eje X hasta que todos los textos queden dentro de la caja de los ejes."""
    fig = ax.figure
    for _ in range(max_iter):
        fig.canvas.draw()
        r = fig.canvas.get_renderer()
        caja = ax.get_window_extent(r)
        x0, x1 = ax.get_xlim()
        rango = x1 - x0
        fuera_der = any(t.get_window_extent(r).x1 > caja.x1 - 4 for t in textos)
        fuera_izq = any(t.get_window_extent(r).x0 < caja.x0 + 4 for t in textos)
        if not (fuera_der or fuera_izq):
            return
        ax.set_xlim(x0 - 0.08 * rango * fuera_izq, x1 + 0.08 * rango * fuera_der)


def graficar_ramas_pci(anio, n_ind=5, extremo="top"):
    p = prod[prod.año == anio].sort_values("pci", ascending=(extremo == "bottom")).head(n_ind).copy()
    p["etiqueta"] = [etiqueta_rama(r) for r in p.itertuples()]
    d = eco[(eco.año == anio) & eco.rama.isin(p.rama)].merge(p[["rama", "etiqueta"]], on="rama")
    pal = "viridis" if extremo == "top" else "rocket"
    fig, axes = plt.subplots(1, 2, figsize=(17, 7), gridspec_kw={"width_ratios": [1, 1.1]})
    sns.barplot(data=p, x="pci", y="etiqueta", hue="etiqueta", palette=pal, legend=False, ax=axes[0])
    axes[0].set_title(T("Nivel de complejidad por rama", "Complexity level by industry"), fontsize=13, fontweight="bold")
    axes[0].set_xlabel(NOM_PCI, fontweight="bold")
    axes[0].set_ylabel("")
    axes[0].axvline(0, color="black", lw=0.8)
    textos = []
    for r in p.itertuples():
        neg = r.pci < 0
        textos.append(axes[0].text(r.pci, r.etiqueta, f"{T('ubicuidad', 'ubiquity')} {r.ubiquity:,}  " if neg
                                   else f"  {T('ubicuidad', 'ubiquity')} {r.ubiquity:,}",
                                   va="center", ha="right" if neg else "left", fontsize=9))
    textos_dentro(axes[0], textos)
    sns.scatterplot(data=d, x="distancia", y="pci", hue="etiqueta", hue_order=list(p.etiqueta), palette=pal, s=40,
                    alpha=0.6, ax=axes[1])
    axes[1].set_title(T("Distancia tecnológica de cada municipio a estas ramas",
                        "Technological distance of each municipality to these industries"), fontsize=13, fontweight="bold")
    axes[1].set_xlabel(NOM_DIST, fontweight="bold")
    axes[1].set_ylabel(NOM_PCI, fontweight="bold")
    axes[1].legend(title=T("Rama", "Industry"), fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
    axes[1].grid(True, ls="--", alpha=0.5)
    fig.tight_layout(w_pad=3)
    texto = T("mayor" if extremo == "top" else "menor", "highest" if extremo == "top" else "lowest")
    titulo = T(f"Las {n_ind} ramas con {texto} complejidad de producto (PCI), México, {anio}",
               f"The {n_ind} industries with the {texto} product complexity (PCI), Mexico, {anio}")
    pie = pie_texto(T("panel izquierdo, PCI de cada rama (ubicuidad = municipios del núcleo con RCA >= 1). Panel derecho, cada "
                      "punto es un municipio; una menor distancia indica más capacidades cercanas a la rama. El PCI de ramas con "
                      "baja ubicuidad es sensible a pocos municipios.",
                      "left panel, PCI of each industry (ubiquity = core municipalities with RCA >= 1). Right panel, each point "
                      "is a municipality; a shorter distance indicates more capabilities close to the industry. The PCI of "
                      "industries with low ubiquity is sensitive to a few municipalities."))
    return finalizar(fig, titulo, pie, gap_pie=GAP_PIE * 1.5)

In [9]:
for anio in ANIOS:
    guardar(graficar_ramas_pci(anio, extremo="top"), f"G04_top5_ramas_pci_{anio}")
    guardar(graficar_ramas_pci(anio, extremo="bottom"), f"G05_bottom5_ramas_pci_{anio}")

Guardado: figuras/ice/G04_top5_ramas_pci_2003.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2003.png
Guardado: figuras/ice/G04_top5_ramas_pci_2008.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2008.png
Guardado: figuras/ice/G04_top5_ramas_pci_2013.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2013.png
Guardado: figuras/ice/G04_top5_ramas_pci_2018.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2018.png
Guardado: figuras/ice/G04_top5_ramas_pci_2023.png
Guardado: figuras/ice/G05_bottom5_ramas_pci_2023.png


## 6. Matriz de diversificación de un municipio entre dos años

Ramas que el municipio no tenía con RCA >= 1 en el año inicial y sí en el final, contra su distancia en el año
inicial. Edita `MUNICIPIOS_DIVERSIFICACION` y `PERIODOS_DIVERSIFICACION` para agregar casos.

In [10]:
def graficar_matriz_diversificacion(clave_mun, anio_ini=2003, anio_fin=2023):
    d0 = eco[(eco.año == anio_ini) & (eco.clave_municipio == clave_mun)]
    d1 = eco[(eco.año == anio_fin) & (eco.clave_municipio == clave_mun)]
    if d0.empty or d1.empty:
        print(f"{clave_mun}: sin datos en {anio_ini} o {anio_fin}")
        return None
    nombre = d1.nom_municipio.iat[0]
    dc = d0[["rama", "pci", "distancia", "rca"]].merge(d1[["rama", "rca"]], on="rama", suffixes=("_ini", "_fin"))
    dc["nueva_ventaja"] = (dc.rca_ini < 1) & (dc.rca_fin >= 1)
    fig, ax = plt.subplots(figsize=(11, 7))
    for flag, color, lab in [(False, "lightgrey", T("Mantuvo o perdió ventaja / sin ventaja", "Kept or lost advantage / no advantage")),
                             (True, "dodgerblue", T("Ganó ventaja (RCA >= 1)", "Gained advantage (RCA >= 1)"))]:
        s = dc[dc.nueva_ventaja == flag]
        ax.scatter(s.distancia, s.pci, c=color, s=80, alpha=0.85, edgecolor="black", lw=0.5, label=lab)
    ax.axvline(dc.distancia.median(), color="firebrick", ls="--",
               label=T(f"Mediana de distancia ({anio_ini})", f"Median distance ({anio_ini})"))
    ax.set_xlabel(T(f"Distancia tecnológica inicial en {anio_ini} (1 - densidad)",
                    f"Initial technological distance in {anio_ini} (1 - density)"), fontweight="bold")
    ax.set_ylabel(f"PCI ({anio_ini})", fontweight="bold")
    ax.legend(title=T(f"Dinámica {anio_ini}-{anio_fin}", f"Dynamics {anio_ini}-{anio_fin}"), loc="upper center",
              bbox_to_anchor=(0.5, -0.12), ncol=3)
    ax.grid(True, ls="--", alpha=0.5)
    titulo = T(f"Matriz de diversificación: adquisición de nuevas ventajas comparativas reveladas, {nombre} ({clave_mun}), México, {anio_ini}-{anio_fin}",
               f"Diversification matrix: acquisition of new revealed comparative advantages, {nombre} ({clave_mun}), Mexico, {anio_ini}-{anio_fin}")
    pie = pie_texto(T(f"cada punto es una rama presente en el núcleo en ambos años. Puntos azules: ramas sin RCA >= 1 en "
                      f"{anio_ini} y con RCA >= 1 en {anio_fin}. El PCI y la distancia son relativos a cada año.",
                      f"each point is an industry present in the core in both years. Blue points: industries without RCA >= 1 in "
                      f"{anio_ini} and with RCA >= 1 in {anio_fin}. PCI and distance are relative to each year."))
    return finalizar(fig, titulo, pie, ref=ax)

In [11]:
MUNICIPIOS_DIVERSIFICACION = ["19039", "20524"]   # Monterrey; Santo Domingo Yodohino
PERIODOS_DIVERSIFICACION = [(2003, 2023)]

for cm in MUNICIPIOS_DIVERSIFICACION:
    for a0, a1 in PERIODOS_DIVERSIFICACION:
        f = graficar_matriz_diversificacion(cm, a0, a1)
        if f is not None:
            guardar(f, f"G06_diversificacion_{cm}_{a0}_{a1}")

Guardado: figuras/ice/G06_diversificacion_19039_2003_2023.png
Guardado: figuras/ice/G06_diversificacion_20524_2003_2023.png


## 7. Distribución del ICE por región CEEY (violín, por año)

In [12]:
def graficar_violin_regiones(anio):
    d = mun[mun.año == anio]
    d = pd.concat([d, d.assign(region_ceey="Nacional")])
    d["region"] = d.region_ceey.map(NOMBRE_REGION)
    orden = [NOMBRE_REGION[r] for r in ORDEN_REGIONES + ["Nacional"]]
    colores = {NOMBRE_REGION[r]: "steelblue" for r in ORDEN_REGIONES} | {NOMBRE_REGION["Nacional"]: "lightgray"}
    fig, ax = plt.subplots(figsize=(11, 6))
    sns.violinplot(data=d, x="region", y="eci", order=orden, hue="region", palette=colores,
                   legend=False, inner="quartile", cut=0, ax=ax)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_xlabel(T("Región CEEY", "CEEY region"), fontweight="bold")
    ax.set_ylabel(NOM_ICE, fontweight="bold")
    ax.grid(axis="y", ls="--", alpha=0.5)
    titulo = T(f"Distribución del Índice de Complejidad Económica municipal por región CEEY, México, {anio}",
               f"Distribution of the municipal Economic Complexity Index by CEEY region, Mexico, {anio}")
    pie = pie_texto(T("las líneas interiores marcan los cuartiles. La línea punteada en 0 es la media del núcleo. "
                      "Regiones según la clasificación del CEEY.",
                      "inner lines mark the quartiles. The dashed line at 0 is the core mean. Regions follow the CEEY "
                      "classification of Mexican states."))
    return finalizar(fig, titulo, pie, ref=ax)

In [13]:
for anio in ANIOS:
    guardar(graficar_violin_regiones(anio), f"G07_violin_regiones_{anio}")

Guardado: figuras/ice/G07_violin_regiones_2003.png
Guardado: figuras/ice/G07_violin_regiones_2008.png
Guardado: figuras/ice/G07_violin_regiones_2013.png
Guardado: figuras/ice/G07_violin_regiones_2018.png
Guardado: figuras/ice/G07_violin_regiones_2023.png


## 8. Complejidad actual (ICE) y potencial de diversificación (COI): Norte vs. Sur (por año)

La leyenda va debajo del área de la gráfica y el recorte es simétrico respecto a los ejes, de modo que el área
principal queda centrada con margen izquierdo suficiente.

In [14]:
def graficar_ice_coi(anio, regiones=("Norte", "Sur"), colores=("dodgerblue", "firebrick")):
    d = mun[(mun.año == anio)]
    fig, ax = plt.subplots(figsize=(10, 7))
    otros = d[~d.region_ceey.isin(regiones)]
    ax.scatter(otros.eci, otros.coi, c="lightgrey", s=20, alpha=0.5, label=T("Otras regiones", "Other regions"))
    for reg, col in zip(regiones, colores):
        s = d[d.region_ceey == reg]
        ax.scatter(s.eci, s.coi, c=col, s=40, alpha=0.7, edgecolor="black", lw=0.3, label=NOMBRE_REGION[reg])
    ax.axvline(0, color="black", ls="--", alpha=0.6, label=T("Media del núcleo (ICE = 0)", "Core mean (ECI = 0)"))
    ax.axhline(d.coi.median(), color="gray", ls="--", alpha=0.6, label=T("Mediana nacional del COI", "National median COI"))
    ax.set_xlabel(NOM_ICE, fontweight="bold")
    ax.set_ylabel(T("Índice de Oportunidad de Complejidad (COI)", "Complexity Outlook Index (COI)"), fontweight="bold")
    ax.legend(title=T("Región CEEY", "CEEY region"), loc="upper center", bbox_to_anchor=(0.5, -0.11), ncol=3)
    ax.grid(True, ls="--", alpha=0.4)
    nombres = T(" vs. ", " vs. ").join(NOMBRE_REGION[r] for r in regiones)
    titulo = T(f"Complejidad actual (ICE) y potencial de diversificación (COI) a nivel municipal, {nombres}, México, {anio}",
               f"Current complexity (ECI) and diversification potential (COI) of municipalities, {nombres}, Mexico, {anio}")
    pie = pie_texto(T("cada punto es un municipio. El COI mide qué tan cerca está un municipio de ramas complejas que todavía "
                      "no tiene; se estandariza con el núcleo. Los demás municipios se muestran en gris.",
                      "each point is a municipality. The COI measures how close a municipality is to complex industries it "
                      "does not yet have; it is standardized with the core. Other municipalities are shown in gray."))
    return finalizar(fig, titulo, pie, ref=ax)

In [15]:
for anio in ANIOS:
    guardar(graficar_ice_coi(anio), f"G08_ice_coi_norte_sur_{anio}")

Guardado: figuras/ice/G08_ice_coi_norte_sur_2003.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2008.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2013.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2018.png
Guardado: figuras/ice/G08_ice_coi_norte_sur_2023.png


## 9. Matrices de transición de quintiles del ICE

Los quintiles se calculan a nivel nacional en cada año sobre los municipios presentes en ambos años.
Orientación de los ejes: eje Y de Q5 (arriba) a Q1 (abajo); eje X de Q1 (izquierda) a Q5 (derecha).
Edita `PERIODOS_TRANSICION` para agregar pares de años.

In [16]:
ETIQ_Q = {1: T("Q1 (más bajo)", "Q1 (lowest)"), 2: "Q2", 3: "Q3", 4: "Q4", 5: T("Q5 (más alto)", "Q5 (highest)")}
ETIQ_PCT = T("Porcentaje de municipios (%)", "Share of municipalities (%)")


def quintiles_movilidad(a0, a1):
    d0 = mun[mun.año == a0][["clave_municipio", "eci", "region_ceey"]]
    d1 = mun[mun.año == a1][["clave_municipio", "eci"]]
    d = d0.merge(d1, on="clave_municipio", suffixes=("_0", "_1")).dropna()
    d["Q0"] = pd.qcut(d.eci_0, 5, labels=False) + 1
    d["Q1"] = pd.qcut(d.eci_1, 5, labels=False) + 1
    return d


def matriz_transicion(d):
    """Filas = quintil de origen (Q5 arriba ... Q1 abajo); columnas = destino (Q1 ... Q5). Filas suman 100."""
    m = pd.crosstab(d.Q0, d.Q1, normalize="index").reindex(index=range(5, 0, -1), columns=range(1, 6), fill_value=0) * 100
    m.index = [ETIQ_Q[q] for q in m.index]
    m.columns = [ETIQ_Q[q] for q in m.columns]
    return m


def heatmap_transicion(d, ax, cbar=True, annot_size=10):
    sns.heatmap(matriz_transicion(d), annot=True, fmt=".1f", cmap="Greens", vmin=0, vmax=100, linewidths=0.5,
                linecolor="white", cbar=cbar, cbar_kws={"label": ETIQ_PCT} if cbar else None,
                annot_kws={"size": annot_size}, ax=ax)
    ax.tick_params(axis="y", rotation=0)


def graficar_transicion_nacional(a0, a1):
    d = quintiles_movilidad(a0, a1)
    fig, ax = plt.subplots(figsize=(9, 7))
    heatmap_transicion(d, ax)
    ax.set_ylabel(T(f"Quintil de origen ({a0})", f"Origin quintile ({a0})"), fontweight="bold")
    ax.set_xlabel(T(f"Quintil de destino ({a1})", f"Destination quintile ({a1})"), fontweight="bold")
    titulo = T(f"Matriz de transición de la complejidad económica municipal, México, {a0}-{a1}",
               f"Transition matrix of municipal economic complexity, Mexico, {a0}-{a1}")
    pie = pie_texto(T(f"las filas suman 100%. n = {len(d):,} municipios presentes en ambos años. Quintiles nacionales del ICE de cada año.",
                      f"rows sum to 100%. n = {len(d):,} municipalities present in both years. National ECI quintiles of each year."))
    return finalizar(fig, titulo, pie)


def graficar_transicion_regional(a0, a1):
    d = quintiles_movilidad(a0, a1)
    regiones = ["Nacional"] + ORDEN_REGIONES
    fig, axes = plt.subplots(2, 3, figsize=(18, 12), layout="constrained")
    fig.get_layout_engine().set(h_pad=0.12, w_pad=0.08, hspace=0.06, wspace=0.03)
    for i, (reg, ax) in enumerate(zip(regiones, axes.flatten())):
        sub = d if reg == "Nacional" else d[d.region_ceey == reg]
        heatmap_transicion(sub, ax, cbar=False, annot_size=13)
        ax.tick_params(labelsize=11.5)
        ax.set_title(f"{NOMBRE_REGION[reg]} (n = {len(sub):,})", fontsize=14, fontweight="bold")
        ax.set_ylabel(T(f"Quintil de origen ({a0})", f"Origin quintile ({a0})") if i % 3 == 0 else "", fontweight="bold",
                      fontsize=13)
        ax.set_xlabel(T(f"Quintil de destino ({a1})", f"Destination quintile ({a1})"), fontweight="bold", fontsize=13)
        if i % 3:
            ax.set_yticklabels([])
    sm = plt.cm.ScalarMappable(cmap="Greens", norm=mcolors.Normalize(0, 100))
    fig.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.7, pad=0.015).set_label(ETIQ_PCT, fontsize=12)
    titulo = T(f"Matrices de transición de la complejidad económica municipal por región CEEY, México, {a0}-{a1}",
               f"Transition matrices of municipal economic complexity by CEEY region, Mexico, {a0}-{a1}")
    pie = pie_texto(T("las filas suman 100%. Los quintiles son nacionales para preservar la comparabilidad entre regiones.",
                      "rows sum to 100%. Quintiles are national to preserve comparability across regions."))
    return finalizar(fig, titulo, pie, tam_titulo=16 * 2, tam_pie=TAM_PIE * 1.5)

In [17]:
PERIODOS_TRANSICION = [(2003, 2023), (2003, 2008), (2008, 2013), (2013, 2018), (2018, 2023)]
for a0, a1 in PERIODOS_TRANSICION:
    guardar(graficar_transicion_nacional(a0, a1), f"G09_transicion_nacional_{a0}_{a1}")
guardar(graficar_transicion_regional(2003, 2023), "G10_transicion_regional_2003_2023")

Guardado: figuras/ice/G09_transicion_nacional_2003_2023.png
Guardado: figuras/ice/G09_transicion_nacional_2003_2008.png
Guardado: figuras/ice/G09_transicion_nacional_2008_2013.png
Guardado: figuras/ice/G09_transicion_nacional_2013_2018.png
Guardado: figuras/ice/G09_transicion_nacional_2018_2023.png
Guardado: figuras/ice/G10_transicion_regional_2003_2023.png


## 10. Mapas municipales

Los polígonos se leen de `data/raw/ice/00mun_2023/00mun.shp` (Marco Geoestadístico 2023, proyección LCC de INEGI)
con `pyshp` y se dibujan con `matplotlib`. Para acelerar, `SIMPLIFICAR` toma uno de cada *k* vértices
(1 = sin simplificar). El título de los mapas mide el doble de la base (`TAM_TITULO_MAPA`) y la imagen se recorta
de forma simétrica alrededor de los polígonos, de modo que el mapa queda centrado.

In [18]:
SIMPLIFICAR = 3
TAM_TITULO_MAPA = TAM_BASE_TITULO * 2


def cargar_poligonos(k=SIMPLIFICAR):
    r = shapefile.Reader(str(C.RAW / "00mun_2023" / "00mun.shp"), encoding="latin-1")
    claves = [rec["CVEGEO"] for rec in r.records()]
    polys, dueños = [], []
    for cve, s in zip(claves, r.shapes()):
        pts = np.asarray(s.points)
        partes = list(s.parts) + [len(pts)]
        for a, b in zip(partes[:-1], partes[1:]):
            anillo = pts[a:b]
            if len(anillo) > 4 * k:
                anillo = np.vstack([anillo[::k], anillo[-1:]])
            polys.append(anillo)
            dueños.append(cve)
    return polys, np.array(dueños)


POLIGONOS, DUEÑOS = cargar_poligonos()
print(f"{len(set(DUEÑOS))} municipios, {len(POLIGONOS):,} anillos")


def dibujar_mapa(ax, valores, cmap, norm=None, colores_cat=None, gris="lightgrey"):
    """valores: Series indexada por clave municipal (numérica o categórica si colores_cat)."""
    v = valores.reindex(DUEÑOS)
    if colores_cat is not None:
        caras = [colores_cat.get(x, gris) for x in v]
    else:
        cm = plt.get_cmap(cmap)
        caras = [cm(norm(x)) if pd.notna(x) else gris for x in v]
    ax.add_collection(PolyCollection(POLIGONOS, facecolors=caras, edgecolors="black", linewidths=0.03))
    ax.margins(0.005)
    ax.autoscale_view()
    ax.set_aspect("equal", anchor="C")
    ax.axis("off")


def barra_color(fig, ax, cmap, norm, etiqueta, ticks=None):
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    cb = fig.colorbar(sm, ax=ax, orientation="horizontal", shrink=0.6, pad=0.02, ticks=ticks)
    cb.set_label(etiqueta, fontsize=11)


def finalizar_mapa(fig, ax, titulo, pie):
    return finalizar(fig, titulo, pie, ref=ax, tam_titulo=TAM_TITULO_MAPA)


NOTA_POLIGONOS = T("Polígonos: Marco Geoestadístico 2023 (INEGI).", "Polygons: INEGI Geostatistical Framework 2023.")

2478 municipios, 2,908 anillos


In [19]:
def graficar_mapa_ice(anio, cmap="RdYlBu", lim=3):
    v = mun[mun.año == anio].set_index("clave_municipio").eci
    norm = mcolors.Normalize(vmin=-lim, vmax=lim)
    fig, ax = plt.subplots(figsize=(12, 9))
    dibujar_mapa(ax, v, cmap, norm)
    barra_color(fig, ax, cmap, norm, T(f"Índice de Complejidad Económica (ICE), escala truncada en ±{lim}",
                                       f"Economic Complexity Index (ECI), scale truncated at ±{lim}"))
    titulo = T(f"Distribución espacial del Índice de Complejidad Económica municipal, México, {anio}",
               f"Spatial distribution of the municipal Economic Complexity Index, Mexico, {anio}")
    pie = pie_texto(NOTA_ICE, T("Gris: municipios sin dato en ese año (no existían o no tenían UE registradas).",
                                "Gray: municipalities without data in that year (not yet created or no registered establishments)."),
                    NOTA_POLIGONOS)
    return finalizar_mapa(fig, ax, titulo, pie)


def mov_estructural(a0, a1):
    d = quintiles_movilidad(a0, a1)
    d["delta_eci"] = d.eci_1 - d.eci_0
    d["salto_quintil"] = d.Q1 - d.Q0
    return d.set_index("clave_municipio")


def graficar_mapa_variacion(a0, a1, cmap="RdBu"):
    d = mov_estructural(a0, a1)
    lim = np.nanpercentile(np.abs(d.delta_eci), 99)
    norm = mcolors.Normalize(vmin=-lim, vmax=lim)
    fig, ax = plt.subplots(figsize=(12, 9))
    dibujar_mapa(ax, d.delta_eci, cmap, norm)
    barra_color(fig, ax, cmap, norm, T(f"Cambio en el ICE estandarizado ({a1} - {a0}); escala truncada en el percentil 99",
                                       f"Change in the standardized ECI ({a1} - {a0}); scale truncated at the 99th percentile"))
    titulo = T(f"Cambio en la posición relativa de complejidad económica municipal, México, {a0}-{a1}",
               f"Change in the relative economic complexity position of municipalities, Mexico, {a0}-{a1}")
    pie = pie_texto(T("el ICE se estandariza en cada año, por lo que el cambio mide posición relativa, no complejidad absoluta. "
                      "Azul: ganó posición; rojo: perdió posición.",
                      "the ECI is standardized in each year, so the change measures relative position, not absolute complexity. "
                      "Blue: gained position; red: lost position."), NOTA_POLIGONOS)
    return finalizar_mapa(fig, ax, titulo, pie)


TIPOLOGIAS = {   # clave interna -> (color, etiqueta es, etiqueta en)
    "liderazgo": ("#08519c", "Liderazgo consolidado (Q4-Q5 en ambos años)", "Consolidated leadership (Q4-Q5 in both years)"),
    "ascendente": ("#31a354", "Movilidad ascendente (subió de quintil)", "Upward mobility (moved up a quintile)"),
    "medio": ("#d9d9d9", "Estancamiento medio (se mantuvo en Q3)", "Middle stagnation (stayed in Q3)"),
    "descendente": ("#fdae61", "Movilidad descendente (bajó de quintil)", "Downward mobility (moved down a quintile)"),
    "rezago": ("#cb181d", "Rezago persistente (Q1-Q2 en ambos años)", "Persistent lag (Q1-Q2 in both years)"),
}


def clasificar_tipologia(q0, q1):
    """Orden de reglas: rezago -> ascendente -> descendente -> liderazgo -> medio."""
    if q0 <= 2 and q1 <= 2:
        return "rezago"
    if q1 > q0:
        return "ascendente"
    if q1 < q0:
        return "descendente"
    if q0 >= 4 and q1 >= 4:
        return "liderazgo"
    return "medio"


def graficar_mapa_tipologias(a0, a1):
    d = mov_estructural(a0, a1)
    tip = pd.Series([clasificar_tipologia(a, b) for a, b in zip(d.Q0, d.Q1)], index=d.index)
    fig, ax = plt.subplots(figsize=(13, 9))
    dibujar_mapa(ax, tip, None, colores_cat={k: v[0] for k, v in TIPOLOGIAS.items()}, gris="black")
    parches = [mpatches.Patch(color=c, label=f"{T(es, en)} (n = {(tip == k).sum():,})") for k, (c, es, en) in TIPOLOGIAS.items()]
    parches.append(mpatches.Patch(color="black", label=T("Sin dato en alguno de los dos años", "No data in one of the two years")))
    ax.legend(handles=parches, loc="lower left", title=T(f"Dinámica de complejidad ({a0}-{a1})", f"Complexity dynamics ({a0}-{a1})"),
              fontsize=9)
    titulo = T(f"Tipología de movilidad de la complejidad económica municipal, México, {a0}-{a1}",
               f"Typology of municipal economic complexity dynamics, Mexico, {a0}-{a1}")
    pie = pie_texto(T("transición entre quintiles nacionales del ICE. Reglas, en orden: Q1-Q2 en ambos años = rezago; subió de "
                      "quintil = ascendente; bajó = descendente; Q4-Q5 en ambos años sin cambio = liderazgo; resto = Q3. Negro: "
                      "municipios creados después del año inicial (p. ej., San Quintín y San Felipe, B.C.).",
                      "transition between national ECI quintiles. Rules, in order: Q1-Q2 in both years = persistent lag; moved up "
                      "= upward; moved down = downward; Q4-Q5 in both years without change = leadership; rest = Q3. Black: "
                      "municipalities created after the initial year (e.g., San Quintín and San Felipe, B.C.)."), NOTA_POLIGONOS)
    return finalizar_mapa(fig, ax, titulo, pie)


def graficar_mapa_salto_quintil(a0, a1, cmap="RdYlGn"):
    d = mov_estructural(a0, a1)
    norm = mcolors.Normalize(vmin=-4, vmax=4)
    fig, ax = plt.subplots(figsize=(12, 9))
    dibujar_mapa(ax, d.salto_quintil, cmap, norm, gris="#f0f0f0")
    barra_color(fig, ax, cmap, norm, T(f"Quintiles ganados o perdidos entre {a0} y {a1}",
                                       f"Quintiles gained or lost between {a0} and {a1}"), ticks=[-4, -2, 0, 2, 4])
    titulo = T(f"Salto de quintil del ICE municipal, México, {a0}-{a1}", f"Quintile jump of the municipal ECI, Mexico, {a0}-{a1}")
    pie = pie_texto(T(f"diferencia entre el quintil nacional del ICE en {a1} y en {a0}. 0 = mismo quintil.",
                      f"difference between the national ECI quintile in {a1} and in {a0}. 0 = same quintile."), NOTA_POLIGONOS)
    return finalizar_mapa(fig, ax, titulo, pie)

In [20]:
for anio in ANIOS:
    guardar(graficar_mapa_ice(anio), f"G11_mapa_ice_{anio}")

PERIODOS_MAPA = [(2003, 2023)]
for a0, a1 in PERIODOS_MAPA:
    guardar(graficar_mapa_variacion(a0, a1), f"G12_mapa_variacion_ice_{a0}_{a1}")
    guardar(graficar_mapa_tipologias(a0, a1), f"G13_mapa_tipologias_{a0}_{a1}")
    guardar(graficar_mapa_salto_quintil(a0, a1), f"G14_mapa_salto_quintil_{a0}_{a1}")

Guardado: figuras/ice/G11_mapa_ice_2003.png
Guardado: figuras/ice/G11_mapa_ice_2008.png
Guardado: figuras/ice/G11_mapa_ice_2013.png
Guardado: figuras/ice/G11_mapa_ice_2018.png
Guardado: figuras/ice/G11_mapa_ice_2023.png
Guardado: figuras/ice/G12_mapa_variacion_ice_2003_2023.png
Guardado: figuras/ice/G13_mapa_tipologias_2003_2023.png
Guardado: figuras/ice/G14_mapa_salto_quintil_2003_2023.png


## 11. Espacio-producto de las ramas (por año)

Red de proximidad entre ramas (φ, calculada con la matriz M del núcleo; ver `export_ecomplexity.py`).
Siguiendo a Hidalgo et al. (2007): árbol de expansión máxima (MST) más los enlaces con φ >= `UMBRAL_PHI`
(regla determinista). El color del nodo es el PCI y el tamaño, el número de UE de la rama en el país.

In [21]:
UMBRAL_PHI = 0.55
SEMILLA_LAYOUT = C.SEED
ue_rama = (pd.read_csv(C.CLEAN / "saic_largo.csv.gz", dtype={"codigo": str}, usecols=["anio", "nivel", "codigo", "variable", "valor"])
             .query("nivel == 'rama' and variable == 'ue'").groupby(["anio", "codigo"]).valor.sum())


def red_espacio_producto(anio, umbral=UMBRAL_PHI):
    e = prox[(prox.año == anio) & (prox.proximidad > 0)]
    G = nx.Graph()
    G.add_weighted_edges_from(e[["rama_1", "rama_2", "proximidad"]].itertuples(index=False, name=None))
    Tr = nx.maximum_spanning_tree(G)
    fuertes = e[e.proximidad >= umbral]
    Tr.add_weighted_edges_from(fuertes[["rama_1", "rama_2", "proximidad"]].itertuples(index=False, name=None))
    return Tr


def graficar_espacio_producto(anio, umbral=UMBRAL_PHI):
    Tr = red_espacio_producto(anio, umbral)
    p = prod[prod.año == anio].set_index("rama")
    pos = nx.spring_layout(Tr, weight="weight", seed=SEMILLA_LAYOUT, k=0.18, iterations=300)
    nodos = list(Tr.nodes())
    pci = p.pci.reindex(nodos).to_numpy()
    tam = np.sqrt(ue_rama.loc[anio].reindex(nodos).fillna(1).to_numpy())
    tam = 15 + 285 * (tam - tam.min()) / (tam.max() - tam.min())
    fig, ax = plt.subplots(figsize=(13, 11))
    nx.draw_networkx_edges(Tr, pos, ax=ax, alpha=0.25, width=0.6, edge_color="gray")
    sc = nx.draw_networkx_nodes(Tr, pos, nodelist=nodos, node_color=pci, cmap="RdYlBu_r", vmin=np.nanmin(pci),
                                vmax=np.nanmax(pci), node_size=tam, edgecolors="black", linewidths=0.3, ax=ax)
    top = p.pci.reindex(nodos).nlargest(8).index
    nx.draw_networkx_labels(Tr, pos, labels={r: r for r in top}, font_size=7, ax=ax)
    fig.colorbar(sc, ax=ax, shrink=0.6).set_label(T("PCI de la rama", "Industry PCI"))
    ax.axis("off")
    titulo = T(f"Espacio-producto de las ramas económicas, México, {anio}", f"Product space of industries, Mexico, {anio}")
    pie = pie_texto(T(f"nodos = ramas del núcleo; enlaces = árbol de expansión máxima de la proximidad φ más los enlaces con "
                      f"φ >= {umbral} ({Tr.number_of_nodes()} nodos, {Tr.number_of_edges()} enlaces). Tamaño proporcional a la "
                      "raíz de las UE de la rama. Se etiquetan las 8 ramas de mayor PCI (código SCIAN).",
                      f"nodes = core industries; edges = maximum spanning tree of proximity φ plus edges with φ >= {umbral} "
                      f"({Tr.number_of_nodes()} nodes, {Tr.number_of_edges()} edges). Node size proportional to the square root "
                      "of the industry's establishments. The 8 industries with the highest PCI are labeled (SCIAN code)."))
    return finalizar(fig, titulo, pie)

In [22]:
for anio in ANIOS:
    guardar(graficar_espacio_producto(anio), f"G15_espacio_producto_{anio}")

Guardado: figuras/ice/G15_espacio_producto_2003.png
Guardado: figuras/ice/G15_espacio_producto_2008.png
Guardado: figuras/ice/G15_espacio_producto_2013.png
Guardado: figuras/ice/G15_espacio_producto_2018.png
Guardado: figuras/ice/G15_espacio_producto_2023.png


## 12. Figuras del Apéndice Metodológico (`figuras/ice/anexo/`)

Dos figuras de diagnóstico que usa `docs/anexo_ice.docx`: censura del PBT por tamaño del municipio y
condicionamiento espectral UE vs. PBT. Leen `outputs/ice/03_censura_pbt.csv` y `outputs/ice/tabla_ue_pbt.csv`
(Módulo 03).

In [23]:
FIG_ANEXO = FIG / "anexo"
FIG_ANEXO.mkdir(exist_ok=True)


def graficar_censura_pbt():
    c = pd.read_csv(C.OUT / "03_censura_pbt.csv")
    q = c[c.desglose == "quintil_ue_mun"].pivot(index="quintil_ue_mun", columns="anio", values="pct_suprimida")
    fig, ax = plt.subplots(figsize=(10, 5.5))
    w, x = 0.15, np.arange(5)
    cols = plt.cm.viridis(np.linspace(0.15, 0.85, len(q.columns)))
    for i, a in enumerate(q.columns):
        ax.bar(x + (i - 2) * w, q[a], w, label=T(f"{a} (Censos {a + 1})", f"{a} (Census {a + 1})"), color=cols[i])
    ax.set_xticks(x)
    ax.set_xticklabels([T("Q1\n(municipios más chicos)", "Q1\n(smallest municipalities)"), "Q2", "Q3", "Q4",
                        T("Q5\n(municipios más grandes)", "Q5\n(largest municipalities)")])
    ax.set_xlabel(T("Quintil del municipio según sus unidades económicas totales",
                    "Municipality quintile by total number of economic units (establishments)"), fontweight="bold")
    ax.set_ylabel(T("Celdas rama × municipio con PBT suprimido (%)",
                    "Industry × municipality cells with suppressed gross output (%)"), fontweight="bold")
    ax.set_ylim(0, 100)
    ax.legend(title=T("Año de referencia (levantamiento censal)", "Reference year (census round)"), ncol=3,
              loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=9.5)
    ax.grid(axis="y", ls="--", alpha=0.5)
    titulo = T("Censura del PBT por confidencialidad según tamaño del municipio, México, 2003-2023",
               "Confidentiality suppression of gross output (PBT) by municipality size, Mexico, 2003-2023")
    pie = pie_texto(T("proporción de celdas con UE > 0 cuyo PBT aparece vacío en SAIC (dato omitido por confidencialidad).",
                      "share of cells with at least one establishment whose gross output (PBT) is blank in SAIC "
                      "(withheld for confidentiality)."))
    return finalizar(fig, titulo, pie, ref=ax)


def graficar_espectro_ue_pbt():
    t = pd.read_csv(C.OUT / "tabla_ue_pbt.csv")
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
    etiquetas = {"UE (núcleo)": T("UE (núcleo)", "Establishments (core)"),
                 "PBT (mismo núcleo)": T("PBT (mismo núcleo)", "Gross output (same core)")}
    for esp, col, mk in [("UE (núcleo)", "navy", "o"), ("PBT (mismo núcleo)", "firebrick", "s")]:
        s = t[t.especificacion == esp].set_index("anio")
        axes[0].plot(s.index, s.lambda2, marker=mk, color=col, label=f"{etiquetas[esp]}: λ2")
        axes[0].plot(s.index, s.lambda3, marker=mk, color=col, ls="--", alpha=0.7, label=f"{etiquetas[esp]}: λ3")
        axes[1].plot(s.index, s.empates_pct, marker=mk, color=col, label=etiquetas[esp])
    axes[0].set_title(T("Segundo y tercer eigenvalor de la matriz M̃", "Second and third eigenvalues of M̃"), fontweight="bold")
    axes[0].set_xlabel(T("Año", "Year"), fontweight="bold")
    axes[0].set_ylabel(T("Eigenvalor", "Eigenvalue"), fontweight="bold")
    axes[0].set_ylim(0, 1.05)
    for a in axes:
        a.set_xticks(ANIOS)
    axes[0].legend(fontsize=8)
    axes[0].grid(ls="--", alpha=0.5)
    axes[1].set_title(T("Municipios con ICE empatado (%)", "Municipalities with tied ECI (%)"), fontweight="bold")
    axes[1].set_xlabel(T("Año", "Year"), fontweight="bold")
    axes[1].set_ylabel("%", fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].grid(ls="--", alpha=0.5)
    fig.tight_layout(w_pad=3)
    titulo = T("Condicionamiento de la matriz de complejidad: UE frente a PBT, México, 2003-2023",
               "Conditioning of the complexity matrix: establishments vs. gross output, Mexico, 2003-2023")
    pie = pie_texto(T("ambos índices se calculan sobre el mismo núcleo de municipios y ramas; en PBT las celdas suprimidas cuentan "
                      "como cero. λ2 = 1 indica una matriz partida en bloques desconectados (2023); λ2 ≈ λ3 indica que el "
                      "eigenvector no está identificado (2018).",
                      "both indices are computed on the same core of municipalities and industries; suppressed gross-output "
                      "cells count as zero. λ2 = 1 indicates a matrix split into disconnected blocks (2023); λ2 ≈ λ3 indicates "
                      "that the eigenvector is not identified (2018)."))
    return finalizar(fig, titulo, pie)


guardar(graficar_censura_pbt(), "anexo/A01_censura_pbt_por_quintil")
guardar(graficar_espectro_ue_pbt(), "anexo/A02_espectro_ue_vs_pbt")

Guardado: figuras/ice/anexo/A01_censura_pbt_por_quintil.png
Guardado: figuras/ice/anexo/A02_espectro_ue_vs_pbt.png


## 13. Inventario de figuras generadas

In [24]:
figs = sorted(str(p.relative_to(FIG)) for p in FIG.rglob("*.png"))
print(f"{len(figs)} figuras en {FIG.relative_to(ROOT)}")
pd.Series(figs).str.extract(r"^(?:anexo/)?([GA]\d+)")[0].value_counts().sort_index()

59 figuras en figuras/ice


0
A01    1
A02    1
G01    6
G02    5
G03    5
G04    5
G05    5
G06    2
G07    5
G08    5
G09    5
G10    1
G11    5
G12    1
G13    1
G14    1
G15    5
Name: count, dtype: int64